# Week 02: Functions, Comprehensions & Data Structures in Depth

## 🎯 Objectives
- Write clean, reusable functions: parameters, return values, defaults, `*args`, `**kwargs`.
- Understand scope and avoid the **mutable default argument** trap.
- Sort, group, and count data fluently (`sorted(key=...)`, `enumerate`, `zip`, `Counter`, `defaultdict`).
- Use slicing, unpacking, and set operations confidently.
- Build a small CSV summary script using only the standard library (pandas comes in Week 6).

## 🔗 Related Resources
- **Roadmap Module:** Module 1 (Python Fundamentals)
- **Algorithm Practice:** Arrays & Hashing (Reverse String, Group Anagrams, Missing Number, Contains Duplicate review)
- **Previous Notebook:** `w01-python-data-structures.ipynb`
- **Next Notebook:** `w03-files-errors-modules.ipynb`

---
## 1. Functions: `return` vs `print`

A function takes inputs (parameters), does work, and **hands a value back** with `return`. `print` only shows something on screen. If a function has no `return`, Python gives back `None`.

This matters in data work: a function that *returns* a value can be reused, tested, and chained. A function that only prints is a dead end.

In [ ]:
def add_print(a, b):
    print(a + b)          # shows the result, returns nothing

def add_return(a, b):
    return a + b          # hands the result back

x = add_print(2, 3)
y = add_return(2, 3)
print(f"x = {x}, y = {y}")           # x is None
print(f"y * 10 = {add_return(2, 3) * 10}")   # only possible because we returned

---
## 2. Parameters: defaults, keywords, `*args`, `**kwargs`

| Feature | Syntax | Meaning |
| :--- | :--- | :--- |
| Default value | `def f(x, y=10)` | `y` is optional |
| Keyword argument | `f(1, y=5)` | Pass by name, order doesn't matter |
| `*args` | `def f(*args)` | Collect extra positional args into a **tuple** |
| `**kwargs` | `def f(**kwargs)` | Collect extra keyword args into a **dict** |

In [ ]:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(greet("Ali"))
print(greet("Ali", greeting="Selamat pagi"))
print(greet(greeting="Hi", name="Ali"))        # keyword order is flexible

def total(*numbers):
    print("numbers is a", type(numbers), numbers)
    return sum(numbers)

def build_record(**attrs):
    print("attrs is a", type(attrs))
    return attrs

print(total(1, 2, 3, 4))
print(build_record(sku="SKU-001", qty=120))

# Unpacking when CALLING a function
nums = [5, 10, 15]
config = {"sku": "SKU-002", "qty": 40}
print(total(*nums))               # same as total(5, 10, 15)
print(build_record(**config))     # same as build_record(sku=..., qty=...)

def mixed(a, b=2, *args, **kwargs):
    print(f"a={a}, b={b}, args={args}, kwargs={kwargs}")

mixed(1)
mixed(1, 5, 6, 7, x=10, y=20)

---
## 3. Scope: Local vs Global

Variables created inside a function are **local**: they disappear when the function ends. Assigning to a name inside a function creates a *new local variable*, even if a global has the same name.

**Best practice:** pass data in through parameters and send results out with `return`. Avoid changing globals from inside functions.

In [ ]:
counter = 0

def set_local():
    counter = 100         # creates a NEW local variable
    return counter

print(set_local())        # 100
print(counter)            # still 0, the global was untouched

def broken_increment():
    try:
        counter += 1      # Python sees an assignment, so 'counter' is local here...
    except UnboundLocalError as e:
        print("UnboundLocalError:", e)   # ...and it has no value yet

broken_increment()

# The clean way: pass in, return out
def increment(value):
    return value + 1

counter = increment(counter)
print(counter)            # 1

---
## 4. The Mutable Default Argument Trap ⚠️

Default values are created **once**, when the function is defined, not each time it's called. If the default is a mutable object (like a list), every call shares the same object. This links directly to Week 1: references and mutability.

**Fix:** use `None` as the default, then create a new list inside the function.

In [ ]:
def add_item_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_item_bad("a"))       # ['a']
print(add_item_bad("b"))       # ['a', 'b']  <- surprise! shared list
print(add_item_bad.__defaults__)

def add_item_good(item, bucket=None):
    if bucket is None:
        bucket = []            # a fresh list on every call
    bucket.append(item)
    return bucket

print(add_item_good("a"))      # ['a']
print(add_item_good("b"))      # ['b']

---
## 5. Docstrings & Type Hints

- A **docstring** (a string right under `def`) explains what a function does. Tools and `help()` read it.
- **Type hints** document what types are expected. Python does **not** enforce them at runtime, but editors and linters use them to catch mistakes.

In [ ]:
def percent_change(old: float, new: float) -> float | None:
    """Return the percent change from old to new.

    Returns None if old is zero (the change is undefined).
    """
    if old == 0:
        return None
    return (new - old) / old * 100

print(percent_change(200, 250))     # 25.0
print(percent_change(0, 10))        # None
print(percent_change.__doc__)

---
## 6. Sorting with `key=` and `lambda`

- `sorted(iterable)` returns a **new sorted list**; `list.sort()` sorts **in place** and returns `None`.
- `key=` takes a function that tells Python *what to sort by*.
- A `lambda` is a tiny anonymous function, handy for `key=`.
- To sort by several criteria, return a **tuple** from the key. A leading `-` flips a number's direction.

In [ ]:
records = [("rice", 120), ("sugar", 45), ("flour", 80), ("salt", 45)]

by_qty = sorted(records, key=lambda r: r[1])
by_qty_desc = sorted(records, key=lambda r: r[1], reverse=True)
print("Ascending: ", by_qty)
print("Descending:", by_qty_desc)

# Sort by qty descending, then name ascending (tie-breaker)
multi = sorted(records, key=lambda r: (-r[1], r[0]))
print("Multi-key: ", multi)

words = ["banana", "Cherry", "apple"]
print(sorted(words))                    # uppercase sorts before lowercase
print(sorted(words, key=str.lower))     # case-insensitive

nums = [3, 1, 2]
result = nums.sort()                    # sorts in place
print(nums, result)                     # result is None, a classic mistake

---
## 7. `enumerate()` and `zip()`

You will use these constantly in data work.

- `enumerate(items)` gives `(index, item)` pairs, so you don't need `range(len(items))`.
- `zip(a, b)` walks two sequences **in parallel**. It stops at the **shortest** one.

In [ ]:
skus = ["A1", "B2", "C3"]
qtys = [10, 20, 30]

for i, sku in enumerate(skus, start=1):
    print(i, sku)

for sku, q in zip(skus, qtys):
    print(f"{sku} -> {q}")

inventory = dict(zip(skus, qtys))       # build a dict from two lists
print(inventory)

print(list(zip([1, 2, 3], ["a", "b"])))     # shortest wins: only 2 pairs

---
## 8. Counting & Grouping: `Counter`, `defaultdict`, `.get()`

| Tool | Use it when |
| :--- | :--- |
| `dict.get(key, default)` | You want a safe lookup without a `KeyError` |
| `collections.Counter` | You need to count how often things appear |
| `collections.defaultdict` | You want to group items without checking "is the key there yet?" |

In [ ]:
from collections import Counter, defaultdict

text = "the quick brown fox jumps over the lazy dog the end"
words = text.split()

# Manual counting with .get()
manual = {}
for w in words:
    manual[w] = manual.get(w, 0) + 1
print(manual)

# Same thing in one line
counts = Counter(words)
print(counts.most_common(3))
print(counts["the"], counts["missing"])    # missing keys give 0, not an error

# Grouping: words by length
by_length = defaultdict(list)
for w in words:
    by_length[len(w)].append(w)
print(dict(by_length))

---
## 9. Slicing & Unpacking

Slicing: `seq[start:stop:step]`. `stop` is **excluded**. Negative indices count from the end. A slice creates a **new list** (a shallow copy, remember Week 1).

In [ ]:
nums = [10, 20, 30, 40, 50]

print(nums[1:4])      # [20, 30, 40]
print(nums[:2])       # [10, 20]
print(nums[-2:])      # [40, 50]
print(nums[::2])      # [10, 30, 50]
print(nums[::-1])     # reversed copy

# Tuple unpacking
a, b = 1, 2
a, b = b, a           # swap without a temp variable
print(a, b)

first, *middle, last = nums
print(first, middle, last)

for name, score in [("ann", 95), ("bo", 80)]:
    print(name, score)

---
## 10. Set Operations

Sets answer "what overlaps?" and "what's different?" very quickly.

| Operation | Operator | Method |
| :--- | :---: | :--- |
| Union | `a \| b` | `a.union(b)` |
| Intersection | `a & b` | `a.intersection(b)` |
| Difference | `a - b` | `a.difference(b)` |
| Symmetric difference | `a ^ b` | `a.symmetric_difference(b)` |

In [ ]:
supplier_a = {"S1", "S2", "S3", "S4"}
supplier_b = {"S3", "S4", "S5"}

print("Either:        ", supplier_a | supplier_b)
print("Both:          ", supplier_a & supplier_b)
print("Only in A:     ", supplier_a - supplier_b)
print("In one, not both:", supplier_a ^ supplier_b)

---
## 11. `map` / `filter` vs Comprehensions

`map` and `filter` exist and you'll see them in other people's code, but **comprehensions are usually clearer** in Python. Prefer a plain loop when the logic gets complicated.

In [ ]:
nums = list(range(1, 11))

# map + filter
old_style = list(map(lambda x: x ** 2, filter(lambda x: x % 2 == 0, nums)))

# comprehension: same result, easier to read
new_style = [x ** 2 for x in nums if x % 2 == 0]

print(old_style)
print(new_style)
print(old_style == new_style)

# Set and dict comprehensions
unique_lengths = {len(w) for w in ["a", "bb", "cc", "ddd"]}
word_lengths = {w: len(w) for w in ["apple", "fig", "banana"]}
print(unique_lengths, word_lengths)

---
## 12. Weekend Build: CSV Summary Script

**Goal:** write `summarize_csv()` using only the standard library. It should:

1. Read the rows into a list of dictionaries (`csv.DictReader`)
2. Print the row count and column names
3. For each numeric column, print **min, max, and average**, ignoring blanks
4. Report columns that contain **missing values**, with counts

The data below is invented. Later, swap in a public dataset or your own anonymized practice file.

In [ ]:
import csv
import io

SAMPLE_CSV = """sku,material,qty,lead_time_days,unit_cost
M-001,Material A,1200,14,0.12
M-002,Material B,,21,0.09
M-003,Material C,860,7,
M-004,Material D,450,30,0.31
M-005,Material E,2300,,0.05
M-006,Material F,150,10,0.44
"""

def to_float(value):
    """Convert a CSV cell to float, or None if it is blank."""
    value = value.strip()
    return float(value) if value else None

# Quick look at how DictReader works
rows = list(csv.DictReader(io.StringIO(SAMPLE_CSV)))
print(rows[0])
print(len(rows), "rows")

In [ ]:
NUMERIC_COLUMNS = ["qty", "lead_time_days", "unit_cost"]

def summarize_csv(file_obj):
    """Print a summary of a CSV file object.

    TODO (your turn):
      1. Read rows with csv.DictReader
      2. Print row count and column names
      3. For each column in NUMERIC_COLUMNS: min, max, average (skip blanks)
      4. Print how many values are missing in each column
    """
    pass

summarize_csv(io.StringIO(SAMPLE_CSV))

# Expected highlights:
# - 6 rows, 5 columns
# - qty: min 150, max 2300, average 992.0 (5 values)
# - missing: qty 1, lead_time_days 1, unit_cost 1

---
## 13. Practice Exercises

Run the helper cell first. Each exercise has a stub and tests. A ❌ means "keep working", a ✅ means it passes.

In [ ]:
def check(label, got, expected):
    status = "✅" if got == expected else "❌"
    print(f"{status} {label}: got {got!r}, expected {expected!r}")

**Exercise 1: `average(nums)`**
Return the mean of a list. Return `None` for an empty list (don't crash).

In [ ]:
def average(nums):
    pass

check("average([1, 2, 3])", average([1, 2, 3]), 2.0)
check("average([10])", average([10]), 10.0)
check("average([])", average([]), None)

**Exercise 2: `word_count(text)`**
Return a dictionary of word frequencies, ignoring case. Write it **manually first** (with `.get()`), then try `Counter`.

In [ ]:
def word_count(text):
    pass

check("word_count('a b a')", word_count("a b a"), {"a": 2, "b": 1})
check("word_count('A a B')", word_count("A a B"), {"a": 2, "b": 1})
check("word_count('')", word_count(""), {})

**Exercise 3: `top_n(d, n)`**
Return the `n` largest items of a dictionary, as a list of `(key, value)` tuples, biggest value first. *Hint: `d.items()` and `sorted(..., key=...)`.*

In [ ]:
def top_n(d, n):
    pass

scores = {"a": 3, "b": 9, "c": 5, "d": 1}
check("top_n(scores, 2)", top_n(scores, 2), [("b", 9), ("c", 5)])
check("top_n(scores, 1)", top_n(scores, 1), [("b", 9)])
check("top_n({}, 3)", top_n({}, 3), [])

**Exercise 4: Rewrite loops as comprehensions**
Below are two loops. Write the equivalent one-line comprehension for each.

In [ ]:
# Loop version 1
squares_of_evens = []
for x in range(1, 21):
    if x % 2 == 0:
        squares_of_evens.append(x ** 2)

# Loop version 2
name_lengths = {}
for name in ["ali", "siti", "muthu", "wei"]:
    name_lengths[name] = len(name)

# Your comprehensions:
squares_of_evens_comp = None
name_lengths_comp = None

check("squares", squares_of_evens_comp, squares_of_evens)
check("name lengths", name_lengths_comp, name_lengths)

**Exercise 5: `sort_by_score(pairs)`**
Given `(name, score)` tuples, return them sorted by score **highest first**. If scores tie, sort by name **A to Z**.

In [ ]:
def sort_by_score(pairs):
    pass

data = [("bo", 80), ("ann", 95), ("cy", 80)]
check("sort_by_score", sort_by_score(data), [("ann", 95), ("bo", 80), ("cy", 80)])
check("sort_by_score([])", sort_by_score([]), [])

**Bonus: `group_by_length(words)`**
Return a dict mapping word length to the list of words with that length. Use `defaultdict(list)`, then convert with `dict(...)`.

In [ ]:
def group_by_length(words):
    pass

check("group_by_length",
      group_by_length(["a", "to", "be", "cat", "go"]),
      {1: ["a"], 2: ["to", "be", "go"], 3: ["cat"]})

---
## 🧠 What Confused Me
*(Fill this in honestly. It's the most useful section when you review later.)*

- 
- 

## 💡 Key Takeaways
*(One or two sentences each, in your own words.)*

1. `return` vs `print`: 
2. Why mutable defaults are dangerous: 
3. When to use `Counter` vs `defaultdict`: 
4. One thing I'd do differently next time: 

## ✅ Week 2 Checklist
- [ ] Sections 1 to 11 read and run
- [ ] Exercises 1 to 5 all show ✅
- [ ] `summarize_csv()` working, committed to GitHub
- [ ] 4 algorithm problems solved and committed (Reverse String, Group Anagrams, Missing Number, plus one of your choice)
- [ ] Progress table updated in `algorithm-practice`
- [ ] "What Confused Me" section filled in